# 응용 3. 확장 칼만 필터로 도는 속도까지 추정해 곡선으로 밀기

**개념:** [개념 4 확장 칼만 필터](../01_concepts/04_extended_kalman_filter.ipynb)

```
응용 1:  AIS 보고 원래 값 ───────────────────────────▶ 직선 외삽
응용 2:  AIS 보고 ──▶ 칼만 필터 (위치·속도) ────────▶ 직선 외삽
응용 3:  AIS 보고 ──▶ 확장 칼만 필터 (+ 도는 속도) ──▶ 곡선 외삽
```

AIS 보고에는 *침로가 도는 속도*가 없다. EKF가 보고 여러 개에서 찾아낸다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sim

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
KN = sim.KN

## 1. 배에 맞게 모델 바꾸기

개념 4의 등속 선회 모델과 같다. 다만 항해에서는 방향을 **북쪽 0°, 시계 방향**으로 재므로 $\sin$, $\cos$ 자리가 바뀐다.

상태: $\mathbf{x} = [e,\ n,\ v,\ \psi,\ \omega]$ (동·북 위치, 속력, 침로, 침로가 도는 속도)

$$\psi' = \psi + \omega\,dt$$
$$e' = e + \frac{v}{\omega}(\cos\psi - \cos\psi'), \qquad n' = n + \frac{v}{\omega}(\sin\psi' - \sin\psi)$$

측정은 AIS 보고의 위치·SOG·COG: $z = [e,\ n,\ v,\ \psi]$. 각도는 359°와 1°의 차이가 2°가 되도록 **감아 줘야**(wrap) 한다.

In [ ]:
def move(x, dt):
    """등속 선회 모델 (북쪽 0, 시계 방향). x = [e, n, v, psi, omega]."""
    e, n, v, psi, w = x
    if abs(w) < 1e-6:
        return np.array([e + v * np.sin(psi) * dt, n + v * np.cos(psi) * dt, v, psi, w])
    psi2 = psi + w * dt
    return np.array([e + v / w * (np.cos(psi) - np.cos(psi2)),
                     n + v / w * (np.sin(psi2) - np.sin(psi)), v, psi2, w])

def jacobian(x, dt, eps=1e-6):
    J = np.zeros((5, 5)); base = move(x, dt)
    for j in range(5):
        d = np.zeros(5); d[j] = eps
        J[:, j] = (move(x + d, dt) - base) / eps
    return J

wrap = lambda a: (a + np.pi) % (2 * np.pi) - np.pi

def ekf_ctrv(rep, q_speed=0.05, q_turn=np.radians(0.05)):
    """등속 선회 EKF. 보고마다 다듬은 상태 [e, n, v, psi, omega] 목록을 돌려준다."""
    sp = sim.POS_NOISE
    H = np.eye(4, 5)
    R = np.diag([sp**2, sp**2, (sim.SOG_NOISE * KN)**2, np.radians(sim.COG_NOISE)**2])
    x = np.array([rep["e"][0], rep["n"][0], rep["sog"][0] * KN, np.radians(rep["cog"][0]), 0.0])
    P = np.diag([sp**2, sp**2, 1.0, np.radians(5)**2, np.radians(1)**2])
    states = []
    for k in range(len(rep["t"])):
        if k > 0:                                                         # 예측
            dt = rep["t"][k] - rep["t"][k - 1]
            J = jacobian(x, dt)
            x = move(x, dt)
            P = J @ P @ J.T + np.diag([0, 0, q_speed**2 * dt, 0, q_turn**2 * dt])
        z = np.array([rep["e"][k], rep["n"][k], rep["sog"][k] * KN, np.radians(rep["cog"][k])])
        y = z - H @ x; y[3] = wrap(y[3])                                   # 보정
        K = P @ H.T @ np.linalg.inv(H @ P @ H.T + R)
        x = x + K @ y; x[3] = wrap(x[3])
        P = (np.eye(5) - K @ H) @ P
        states.append(x.copy())
    return np.array(states)

## 2. 도는 속도 추정

`sim.py`가 정답을 아는 가짜 배를 만든다.

- 12노트로 북쪽으로 가다가 **200~380초에 1°/초로 우회전**(180° 돌기), 그 뒤 다시 직진
- **20초마다** AIS 보고. 위치 ±5m, SOG ±0.2kn, COG ±2° 잡음

앱은 보고를 받을 때마다 그 값으로 다음 보고 전까지 배를 밀어 그린다. 이 *민 위치*가 실제와 몇 m 틀리는지 직진·선회 구간별로 잰다.

In [ ]:
rep = sim.make_reports()
ekf = ekf_ctrv(rep)

fig, ax = plt.subplots(figsize=(9, 3.5))
t = np.arange(0, sim.DURATION + 1, 1)
ax.plot(t, sim.turn_rate_at(t), "k:", label="실제 도는 속도")
ax.plot(rep["t"], np.degrees(ekf[:, 4]), "o-", label="EKF 추정 (보고에 없는 값!)")
ax.set_xlabel("초"); ax.set_ylabel("°/초"); ax.grid(alpha=.3); ax.legend(); plt.show()

선회가 시작되고 보고 한두 개가 지나서야 알아챈다. 끝날 때도 조금 늦게 0으로 돌아온다.

## 3. 세 방법 비교

In [ ]:
def kalman_cv(rep, q=0.05):
    """등속 칼만 필터. 보고마다 다듬은 상태 [e, n, ve, vn] 목록을 돌려준다."""
    sp = sim.POS_NOISE
    H = np.eye(4)
    x = np.array([rep["e"][0], rep["n"][0], 0.0, 0.0])
    P = np.diag([sp**2, sp**2, 10.0, 10.0])
    states = []
    for k in range(len(rep["t"])):
        if k > 0:                                                  # 예측 (선형 외삽)
            dt = rep["t"][k] - rep["t"][k - 1]
            F = np.eye(4); F[0, 2] = F[1, 3] = dt
            G = np.array([[dt**2 / 2, 0], [0, dt**2 / 2], [dt, 0], [0, dt]])
            x = F @ x
            P = F @ P @ F.T + q * G @ G.T
        v, c = rep["sog"][k] * KN, np.radians(rep["cog"][k])       # 보정
        z = np.array([rep["e"][k], rep["n"][k], v * np.sin(c), v * np.cos(c)])
        sv = np.hypot(sim.SOG_NOISE * KN, v * np.radians(sim.COG_NOISE))
        R = np.diag([sp**2, sp**2, sv**2, sv**2])
        K = P @ H.T @ np.linalg.inv(H @ P @ H.T + R)
        x = x + K @ (z - H @ x)
        P = (np.eye(4) - K @ H) @ P
        states.append(x.copy())
    return np.array(states)

kf = kalman_cv(rep)

def predict_kf(k, tau):
    e, n, ve, vn = kf[k]
    return e + ve * tau, n + vn * tau

def predict_ekf(k, tau):
    path = np.array([move(ekf[k], s)[:2] for s in tau])
    return path[:, 0], path[:, 1]

def predict_dr(k, tau):
    """보고 k 의 원래 SOG·COG로 tau 초 곧게 민 위치."""
    v, c = rep["sog"][k] * KN, np.radians(rep["cog"][k])
    return rep["e"][k] + v * np.sin(c) * tau, rep["n"][k] + v * np.cos(c) * tau

methods = [("직선 DR (응용 1)", predict_dr, "C3"), ("칼만 필터 (응용 2)", predict_kf, "C0"), ("EKF + 곡선 (응용 3)", predict_ekf, "C2")]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sim.plot_truth(ax, rep)
for name, f, color in methods:
    sim.plot_extrapolation(ax, rep, f, name, color)
(xl, yl) = sim.ZOOM_VIEWS[1]; ax.set_xlim(*xl); ax.set_ylim(*yl); ax.legend(loc="lower center"); ax.set_title("선회 구간"); plt.show()

results = {name: sim.evaluate(rep, f) for name, f, _ in methods}
for name, r in results.items():
    print(f"{name:16s} 직진 {r['직진']:5.2f} m   선회 {r['선회']:5.2f} m")

fig, ax = plt.subplots(figsize=(7, 3.5))
w = 0.25
for i, (name, _, color) in enumerate(methods):
    ax.bar(np.arange(2) + (i - 1) * w, [results[name]["직진"], results[name]["선회"]], w, label=name, color=color)
ax.set_xticks([0, 1], ["직진", "선회"]); ax.set_ylabel("평균 오차 (m)"); ax.legend(); ax.grid(axis="y", alpha=.3); plt.show()

## 정리

| | 응용 1 직선 DR | 응용 2 칼만 필터 | 응용 3 확장 칼만 필터 |
|---|---|---|---|
| 외삽 모양 | 직선 | 직선 | **곡선** (원호) |
| 외삽에 넣는 값 | 보고 원래 값 | 다듬은 위치·속도 | 다듬은 위치·속도·침로 + **도는 속도** |
| 모델 | 등속 직선 | 등속 직선 | 등속 선회 |
| 잘하는 것 | 단순, 계산 가벼움 | 잡음 줄이기 | 잡음 + 선회 |
| 약점 | 잡음, 선회 | 선회 | 선회 시작·끝을 늦게 알아챔, 조정할 값이 많음 |

세 방법 모두 **외삽**이고, 넓게 보면 모두 **추측항법**이다. 차이는 *가정(모델)*을 얼마나 실제에 맞게 만들고, 거기에 넣을 값을 얼마나 잘 다듬느냐다.

## 직접 해 보기

1. `ekf_ctrv(rep, q_turn=np.radians(0.5))`처럼 `q_turn`을 키우면 도는 속도를 더 빨리 알아채나? 직진 오차는 어떻게 되나?
2. `sim.TURN_RATE = 3.0`(급선회)으로 바꾸면 세 방법의 차이는 커지나?
3. `sim.REPORT_INTERVAL = 5`로 줄이면 세 방법의 차이는 어떻게 되나? 왜 그런가?

In [ ]:
# 여기서 직접 바꿔 보세요
